# Full-row temporal feature selection

This executed example uses synthetic development data with signal, redundancy, interactions, a small business segment, and drift. It demonstrates memetic NSGA-II, RFE, local refinement, reports, selection, full-development refitting, and checkpoint recovery. It never accesses independent OOT data. Install with `python -m pip install -e ".[dev]"` before running.

In [1]:
from pathlib import Path
import tempfile
from IPython.display import display
from memetic_fs import (MemeticNSGA2Selector, OptimizationConfig, ResourceConfig, SelectionConfig)
from memetic_fs.synthetic import make_synthetic_data

root = Path.cwd()
if root.name == "examples":
    root = root.parent
(root / "runs").mkdir(exist_ok=True)
run_dir = Path(tempfile.mkdtemp(prefix="notebook_", dir=root / "runs"))
frame = make_synthetic_data(n_rows=150000, 
                            n_features=100,
                            n_months=10, 
                            random_state=42, 
                            n_signal_features=28,
                            n_redundant_features=4,
                            n_interaction_features=4,
                            n_segmentation_specific_features=6)
display(frame.head())

,signal_0,signal_1,signal_2,signal_3,signal_4,signal_5,signal_6,signal_7,signal_8,signal_9,...,noise_53,noise_54,noise_55,noise_56,noise_57,target,reference_month,applicant_id,customer_type,age_group
0,0.304717,-1.039984,0.750451,0.940565,-1.951035,-1.302179,0.127840,-0.316243,-0.016801,-0.853044,...,-1.447112,-1.322700,-0.997247,0.399774,-0.905479,0,202301,0,client,60+
1,-0.378163,1.299228,-0.356264,0.737516,-0.933618,-0.205438,-0.950022,-0.339033,0.840308,-1.727320,...,-0.894727,0.643327,-0.394605,-0.005122,-0.163443,1,202301,1,client,46-60
2,0.337575,1.407482,0.090585,0.643939,-2.050172,-0.048718,-0.843230,-1.218813,-0.878152,-0.334123,...,0.176512,0.295994,-0.371915,-1.756722,0.327995,1,202301,2,client,46-60
3,1.727350,-1.533861,0.863828,-0.328525,-0.061324,-1.052899,-0.334456,1.300045,0.582655,1.732312,...,0.138241,0.760133,0.229211,0.530065,-0.704673,1,202301,3,client,46-60
4,-0.179611,0.196776,0.820528,-0.393741,0.521167,-0.265839,-0.117542,0.829519,-1.993060,-1.296472,...,0.935547,-1.831406,-0.335607,-1.990812,-1.495061,0,202301,4,client,31-45


Every observed month is complete. Default CV trains on the first three months and expands; each validation period follows training. Metadata is excluded from predictors, IDs must be unique, and the small prospect segment remains descriptive when its event support is insufficient for a configured hard constraint.

Progress is shown throughout the run with timestamped logs. Set `verbose=2` for candidate/fold detail or `verbose=0` for quiet console output. Detailed parent and worker files are always saved under the experiment `logs/` directory; long native fits emit periodic heartbeats.

In [2]:
selector = MemeticNSGA2Selector(
    time_cv="reference_month", 
    feature_range=(10, 90),
    num_boost_round=100, 
    model_params=None,#{"num_leaves": 7, "min_data_in_leaf": 10},
    optimization=OptimizationConfig(population_size=100, 
                                    max_generations=30,
                                    min_generations=10, 
                                    patience=10, 
                                    local_search_interval=5,
                                    local_search_budget=50, 
                                    rfe_seed_sizes=(5,10,15,20,25,30,35,40,45,50,55,60,65,70,75,80,85,90),
                                    reduced_generations=10),
    resources=ResourceConfig(max_workers=15, threads_per_worker=1),
    selection=SelectionConfig(gini_tolerance=0.01),
    checkpoint_dir=run_dir, random_state=42,
)
selector.fit(frame, 
             "target", 
             id_col="applicant_id",
             segment_cols=["customer_type", "age_group"])
display(selector.pareto_front_[["key", "n_features", "mean_gini", "std_gini", "min_gini"]])
print("Selected:", selector.best_features_)
print("Stopped:", selector.stop_reason_)

2026-11-21T20:25:07Z INFO pid=109964 Validating development data | columns=105 phase="data" rows=150000
2026-11-21T20:25:07Z INFO pid=109964 Development data validated | categorical_features=0 observed_months=10 phase="data" predictors=100 rows=150000
2026-11-21T20:25:08Z INFO pid=109964 Temporal folds validated | folds=7 phase="cv" training_rows=[45000, 60000, 75000, 90000, 105000, 120000, 135000] validation_rows=[15000, 15000, 15000, 15000, 15000, 15000, 15000]
2026-11-21T20:25:08Z INFO pid=109964 Read-only experiment data persisted | data_bytes=67230000 phase="storage"
2026-11-21T20:25:08Z INFO pid=109964 Evaluator initialized | active_constraints=0 candidate_features=100 excluded_constraints=0 folds=7 identity="478047057539" n_rows=150000 training_threads=1
2026-11-21T20:25:08Z INFO pid=109964 Full-feature reference calibration started | folds=7 n_features=100 num_boost_round=100
2026-11-21T20:25:08Z INFO pid=109964 Evaluation batch prepared | cache_hits=0 duplicate_requests=0 feat

,key,n_features,mean_gini,std_gini,min_gini
0,207e7c089a6d56563380f06b49ca38bb53f89a2c7e8b4c...,10,0.810080,0.004304,0.802813
1,92b9d3f083102c8a7b461218e8309a21d7ae9435db6ea7...,11,0.810889,0.003665,0.804959
2,7b1c6410b7dc85ad3a5accb475d11264c2bc56a2aad9ab...,12,0.811635,0.006380,0.801111


Selected: ['signal_0', 'signal_1', 'redundant_3', 'interaction_0', 'interaction_1', 'interaction_2', 'interaction_3', 'segment_signal_1', 'segment_signal_3', 'noise_56']
Stopped: max_generations


Final comparisons and convergence use full canonical CV. Reduced-fold results have a separate provisional archive. The objective is equal-fold mean Gini (`2*AUC-1`); pooled out-of-fold Gini and temporal stability are diagnostics.

In [3]:
display(selector.temporal_performance_)
display(selector.segment_performance_.head(10))
display(selector.selection_frequency_.head(12))
display(selector.history_)
assert {r["fidelity"] for r in selector.confirmed_} == {"full"}
assert all(len(r["folds"]) == len(selector.folds_) for r in selector.confirmed_)
assert selector.history_["local_search_evaluations"].sum() > 0
assert selector.rfe_trajectory_

,auc,events,fold,gini,n,nonevents,train_n,validation_n
0,0.905553,6945,2023-04,0.811106,15000,8055,45000,15000
1,0.905390,6813,2023-05,0.810779,15000,8187,60000,15000
2,0.907935,6802,2023-06,0.815871,15000,8198,75000,15000
3,0.906330,6948,2023-07,0.812660,15000,8052,90000,15000
4,0.906306,6890,2023-08,0.812612,15000,8110,105000,15000
5,0.902361,6888,2023-09,0.804721,15000,8112,120000,15000
6,0.901407,6801,2023-10,0.802813,15000,8199,135000,15000


,auc,column,events,fold,gini,n,nonevents,value
0,0.913265,customer_type,6061,2023-04,0.826529,13125,7064,client
1,0.853574,customer_type,884,2023-04,0.707147,1875,991,prospect
2,0.900832,age_group,1651,2023-04,0.801664,3598,1947,60+
3,0.906949,age_group,1777,2023-04,0.813897,3824,2047,46-60
4,0.904451,age_group,1775,2023-04,0.808902,3792,2017,31-45
5,0.909800,age_group,1742,2023-04,0.819599,3786,2044,18-30
6,0.914719,customer_type,5942,2023-05,0.829438,13125,7183,client
7,0.839497,customer_type,871,2023-05,0.678995,1875,1004,prospect
8,0.904447,age_group,1727,2023-05,0.808894,3791,2064,60+
9,0.909490,age_group,1713,2023-05,0.818980,3768,2055,46-60


,feature,count,frequency,n_solutions
0,interaction_0,3,1.000000,3
1,interaction_1,3,1.000000,3
2,interaction_2,3,1.000000,3
3,interaction_3,3,1.000000,3
4,redundant_3,3,1.000000,3
5,segment_signal_1,3,1.000000,3
6,segment_signal_3,3,1.000000,3
7,signal_0,3,1.000000,3
8,signal_1,3,1.000000,3
9,segment_signal_5,2,0.666667,3


,best_confirmed_gini,confirmed_count,confirmed_hypervolume,converged,feature_count_max,feature_count_mean,feature_count_min,fidelity,generation,local_search_evaluations,provisional_count,stalled_generations,unique_evaluations
0,0.806335,1,0.830983,False,90,41.33,10,last_1,1,0,2,0,119
1,0.806335,1,0.830983,False,90,26.90,10,last_1,2,0,2,1,219
2,0.806335,1,0.830983,False,89,20.74,10,last_1,3,0,2,2,320
3,0.806335,1,0.830983,False,60,17.01,10,last_1,4,0,2,3,420
4,0.809861,3,0.832574,False,34,14.13,10,last_1,5,50,2,0,622
5,0.809861,3,0.832574,False,34,13.49,10,last_1,6,0,2,1,722
6,0.809861,3,0.832580,False,34,13.19,10,last_1,7,0,2,2,823
7,0.809861,3,0.832580,False,34,12.88,10,last_1,8,0,2,3,923
8,0.809861,3,0.832580,False,28,12.69,10,last_1,9,0,3,4,1022
9,0.809861,3,0.832581,False,20,12.34,10,last_1,10,50,3,5,1214


The offline report includes the Pareto approximation, cardinality/RFE trajectory, feature frequency, co-selection, temporal Gini, segment metrics, and evolution history. CSV/Parquet/JSON exports accompany it.

In [7]:
from memetic_fs.persistence.checkpoints import read_json
from memetic_fs.reporting.visualizations import build_figures
payload = read_json(run_dir / "results.json")
figures = build_figures(payload)
print("Charts:", list(figures))
print("Offline report:", run_dir / "report.html")
print("Exports:", sorted(p.name for p in run_dir.glob("*.parquet")))

Charts: ['frontier', 'cardinality', 'frequency', 'coselection', 'temporal', 'segments', 'history']
Offline report: c:\Users\tms19\OneDrive\Documents\ChatGPT\Memetic NSGA-II Feature Selection Framework\runs\notebook_v8t0_8o3\report.html
Exports: ['confirmed_masks.parquet', 'evaluations.parquet', 'fold_performance.parquet', 'high_performing_frequency.parquet', 'history.parquet', 'rfe_trajectory.parquet', 'segment_constraints.parquet', 'segment_performance.parquet', 'selection_frequency.parquet']


Selection policies include tolerance, geometric knee, manual candidate choice, and an approximate paired temporal-block uncertainty rule. With few validation folds, statistical evidence is weak. Here the chosen manual candidate is then refitted on all development rows. External generalization assessment remains separate.

In [8]:
selected_id = selector.selected_candidate_["key"]
assert selector.select("manual", candidate_id=selected_id)
selected_data = selector.transform(frame)
model = selector.fit_final_model()
assert model.num_feature() == selected_data.shape[1]
print("Final model features:", selected_data.shape[1])
print("Final model:", run_dir / "final_model.txt")

2026-11-21T19:08:47Z INFO pid=109108 Final candidate selected | candidate="1b409157eea6" features=4 mean_gini=0.8146842909741998 phase="selection" policy="manual"
2026-11-21T19:08:47Z INFO pid=109108 Exporting experiment results | interactive_plots=true phase="reporting"
2026-11-21T19:08:48Z INFO pid=109108 Result artifacts saved | directory="C:\\Users\\tms19\\OneDrive\\Documents\\ChatGPT\\Memetic NSGA-II Feature Selection Framework\\runs\\notebook_v8t0_8o3" phase="reporting"
2026-11-21T19:08:48Z INFO pid=109108 Checkpoint saved | generation=12 phase="checkpoint"
2026-11-21T19:08:48Z INFO pid=109108 Fitting final model on all development rows | features=4 phase="final_model" rounds=100 rows=15000
2026-11-21T19:08:48Z INFO pid=109108 Final development model saved | elapsed_seconds=0.119 phase="final_model"


Final model features: 4
Final model: c:\Users\tms19\OneDrive\Documents\ChatGPT\Memetic NSGA-II Feature Selection Framework\runs\notebook_v8t0_8o3\final_model.txt


Resume verifies persisted input digests and evaluator settings. Completed model evaluations are reused. JSON optimizer checkpoints avoid executable pickle deserialization.

In [6]:
completed_before = len(selector.evaluator_.cache.completed())
restored = MemeticNSGA2Selector.resume(run_dir)
assert len(restored.evaluator_.cache.completed()) == completed_before
assert [r["mask"] for r in restored.confirmed_] == [r["mask"] for r in selector.confirmed_]
print("Reused completed evaluations:", completed_before)
print("Confirmed frontier size:", len(restored.confirmed_))

2026-11-21T11:34:15Z INFO pid=100692 Checkpoint inputs and dependency versions verified | directory="C:\\Users\\tms19\\OneDrive\\Documents\\ChatGPT\\Memetic NSGA-II Feature Selection Framework\\runs\\notebook_ad5xquc7" phase="resume"


2026-11-21T11:34:15Z INFO pid=100692 Evaluator initialized | active_constraints=0 candidate_features=12 excluded_constraints=0 folds=5 identity="c40d4b2d3395" n_rows=640 training_threads=1


2026-11-21T11:34:15Z INFO pid=100692 Full-feature reference calibration started | folds=5 n_features=12 num_boost_round=10


2026-11-21T11:34:15Z INFO pid=100692 Evaluation batch prepared | cache_hits=1 duplicate_requests=0 feature_counts=[12] fidelity="full" new_evaluations=0 pending_duplicates=0 reference_reuses=0 requested_candidates=1 unique_candidates=1


2026-11-21T11:34:15Z INFO pid=100692 Evaluation batch completed | cache_hits=1 elapsed_seconds=0.03761749999830499 evaluated_candidates=0 fidelity="full" requested_candidates=1


2026-11-21T11:34:15Z INFO pid=100692 Full-feature reference calibration completed | elapsed_seconds=0.03852520001237281 estimated_task_bytes=205084672 mean_gini=0.7029576791607625 measured_private_peak_bytes=157757440


... (10 outputs left out)

2026-11-21T11:34:15Z INFO pid=100692 Result artifacts saved | directory="C:\\Users\\tms19\\OneDrive\\Documents\\ChatGPT\\Memetic NSGA-II Feature Selection Framework\\runs\\notebook_ad5xquc7" phase="reporting"


2026-11-21T11:34:15Z INFO pid=100692 Checkpoint saved | generation=4 phase="checkpoint"


2026-11-21T11:34:15Z INFO pid=100692 Feature selection completed | confirmed_pareto_candidates=4 elapsed_seconds=0.298 log="C:\\Users\\tms19\\OneDrive\\Documents\\ChatGPT\\Memetic NSGA-II Feature Selection Framework\\runs\\notebook_ad5xquc7\\logs\\run.log" phase="complete" selected_features=6 stop_reason="max_generations"


Reused completed evaluations: 50
Confirmed frontier size: 4


Repeated subset search creates model-selection optimism. This frontier describes evaluated candidates and has no global-optimality guarantee. Run independent OOT assessment outside this framework. For production scale, use the separate `benchmarks/full_row.py` utility; this notebook does not establish feasibility for 1.5 million rows.